In [10]:
import json
import time
import requests

MODEL = "qwen3:4b"
OLLAMA_URL = "http://localhost:11434/api/generate"

with open("FinQA/dataset/dev.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} FinQA dev entries")

Loaded 883 FinQA dev entries


In [11]:
# This must finish in a few seconds. If it hangs or errors, the problem
# is Ollama/plumbing, NOT the FinQA workload.

try:
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": "What is 2+2? Answer with only the number.",
            "stream": False,
            "think": False,
            "options": {"temperature": 0, "num_ctx": 8192}
        },
        timeout=120
    )
    print("Status:", response.status_code)
    result = response.json()
    print("Answer:", result.get("response", "").strip())
    print("Error field (if any):", result.get("error", "none"))

    gen_tokens = result.get("eval_count", 0)
    gen_time = result.get("eval_duration", 1) / 1e9
    print(f"Speed: {gen_tokens} tokens in {gen_time:.1f}s = {gen_tokens/max(gen_time,0.001):.1f} tok/s")

except requests.exceptions.Timeout:
    print("TIMED OUT after 120s — Ollama is stuck or unreachable.")
except Exception as e:
    print("FAILED:", e)

Status: 200
Answer: Okay, the user asked "What is 2+2? Answer with only the number." Hmm, this seems straightforward but let me think carefully. 

First, I recall that 2 plus 2 is a basic arithmetic problem. In elementary math, 2 + 2 equals 4. The user specifically said to answer with only the number, so I shouldn't add any extra text or explanations. 

I wonder why they're asking this though. Maybe it's a test to see if I follow instructions precisely? Or perhaps they're checking if I'll overcomplicate simple questions. The phrasing "Answer with only the number" suggests they want minimal response - no "4" with spaces or anything. 

Also, the user seems to be in a hurry or just wants quick confirmation. No emotional cues in the query, so I'll keep it neutral. Better double-check: 2 + 2 = 4 is universally accepted math. No tricks here. 

Final decision: Just output "4" as instructed. No extra characters. Strictly following their request.
</think>

4
Error field (if any): none
Speed: 22

In [12]:
def ask_model(prompt, model=MODEL):
    """Send a prompt to Ollama, return (answer, stats_dict)."""
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "think": False,
            "options": {
                "temperature": 0,
                "num_ctx": 8192,
                "num_predict": 200   # hard cap on output length — stops rambling
            }
        },
        timeout=600
    )
    result = response.json()

    if "error" in result:
        raise RuntimeError(f"Ollama error: {result['error']}")

    stats = {
        "prompt_tokens": result.get("prompt_eval_count", 0),
        "prompt_seconds": result.get("prompt_eval_duration", 0) / 1e9,
        "gen_tokens": result.get("eval_count", 0),
        "gen_seconds": result.get("eval_duration", 0) / 1e9,
    }
    stats["tok_per_sec"] = stats["gen_tokens"] / max(stats["gen_seconds"], 0.001)

    return result["response"].strip(), stats

In [13]:
def build_prompt(entry):
    pre = "\n".join(entry["pre_text"])
    post = "\n".join(entry["post_text"])
    table = "\n".join(" | ".join(cell for cell in row) for row in entry["table"])

    prompt = f"""You are a financial analyst. Read the following excerpt from a financial report and answer the question.

{pre}

TABLE:
{table}

{post}

QUESTION: {entry["qa"]["question"]}

Respond with ONLY the final numeric answer. No explanation, no working, no units unless asked."""
    return prompt

In [14]:
results = []

for i in range(5):
    entry = data[i]
    prompt = build_prompt(entry)

    start = time.time()
    try:
        answer, stats = ask_model(prompt)
    except Exception as e:
        print(f"--- Question {i} FAILED: {e} ---\n")
        continue
    elapsed = time.time() - start

    print(f"--- Question {i} ({elapsed:.1f}s | {stats['gen_tokens']} tok @ {stats['tok_per_sec']:.1f} tok/s) ---")
    print(f"Q:     {entry['qa']['question']}")
    print(f"Model: {answer}")
    print(f"Gold:  {entry['qa']['answer']}")
    print()

    results.append({
        "index": i,
        "question": entry["qa"]["question"],
        "model_answer": answer,
        "gold_answer": entry["qa"]["answer"],
        "seconds": round(elapsed, 1),
        "gen_tokens": stats["gen_tokens"],
        "tok_per_sec": round(stats["tok_per_sec"], 1),
    })

--- Question 0 (5.9s | 200 tok @ 66.2 tok/s) ---
Q:     what is the average payment volume per transaction for american express?
Model: First, I need to find the average payment volume per transaction for American Express. The question is: "what is the average payment volume per transaction for american express?"

I have the table from the financial report. Let me look at the data for American Express.

The table shows:

- Company: american express

- Payments volume (billions): 637

- Total volume (billions): 647

- Total transactions (billions): 5.0

- Cards (millions): 86

The question is about "average payment volume per transaction". I think "payment volume" here might be a bit confusing. In the context, "payments volume" probably refers to the total amount of money processed, like in billions of dollars.

Let me read the question carefully: "average payment volume per transaction". But in the table, there's "payments volume" and "total transactions".

I recall that in payment pro

In [15]:
with open("results_02_first5.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to results_02_first5.json")

Saved to results_02_first5.json
